# 05 · Do the embeddings help a fraud model?

**NVIDIA TFM Blueprint, notebook 5 of 5.** v1's closing question: train the
same classifier on raw features, then on raw features plus embeddings, and
compare. If pretraining learned something useful, the combined model beats the
raw one.

Every notebook runs on CPU in about a minute on a small generated book, so
you can read the whole series without downloading anything. To see real rows,
build the TabFormer splits once (notebook 01 shows how) and set
`TFM_TABFORMER_ROOT`; the notebooks then load 24 TabFormer users with
`config.small.yaml`. The full-size runs are recipe commands, listed in notebooks 03
and 05, not notebook cells.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tfm.data.collate import minibatches
from tfm.evaluation.fraud.baselines import fit_predict_probe
from tfm.evaluation.fraud.metrics import classification_metrics
from tfm.evaluation.tabular import TabularFeaturizer
from tfm.inference.embedding import extract_embeddings
from tfm.runtime import build_model, resolve_device
from tfm.training.trainer import train_model

from tfm.config import load_recipe_config
from tfm.data.adapters.paths import ADAPTER_ENV
from tfm.extras.capability import extra_available
from tfm.runtime import prepare_data
from tfm.runtime.common import recipe_dir
from tfm.training.reproducibility import seed_everything

USING_TABFORMER = bool(os.environ.get(ADAPTER_ENV["tabformer"]))
CONFIG = recipe_dir("blueprint_v1") / ("config.small.yaml" if USING_TABFORMER else "config.smoke.yaml")
config = load_recipe_config(CONFIG)
if config.model.backbone == "huggingface" and not extra_available("transformers"):
    print("transformers extra not installed; using the eager decoder (uv sync --extra huggingface)")
    composition = config.model.composition.model_copy(update={"position": "nope"})
    model_cfg = config.model.model_copy(update={"backbone": "eager", "composition": composition})
    config = config.model_copy(update={"model": model_cfg})
seed_everything(config.seed)
prepared = prepare_data(config)
tokenizer = prepared.tokenizer
train_batch, val_batch, test_batch = prepared.batches
print("config", CONFIG.name, "| adapter", config.data.adapter, "| backbone", config.model.backbone)
from tfm.runtime.evaluate import event_entity_split, history_window_tasks

split = event_entity_split(prepared, config.seed)
tasks = history_window_tasks(prepared, config, split)
rows, control = tasks["history_window"], tasks["single_event"]

In [ ]:
device = resolve_device(config.runtime.device)


def pretrain(seed):
    seed_everything(seed)
    model = build_model(config, tokenizer.vocab_size)
    train_model(
        model, minibatches(train_batch, config.training.batch_size), max_steps=config.training.max_steps,
        learning_rate=config.training.learning_rate, seed=seed, device=device,
    )
    return model


def embed(model):
    return {
        "history": extract_embeddings(model, rows.train_batch, device=device).vectors,
        "history_eval": extract_embeddings(model, rows.eval_batch, device=device).vectors,
        "single": extract_embeddings(model, control.train_batch, device=device).vectors,
        "single_eval": extract_embeddings(model, control.eval_batch, device=device).vectors,
    }


model = pretrain(config.seed)
emb = embed(model)
print({k: v.shape for k, v in emb.items()})

## 1. Four feature sets, one head

Four models with the same head, fit on the training rows and scored on the
test rows of notebook 01. Only the features differ:

| arm | features |
|---|---|
| raw | the transaction's own fields (v1's raw arm) |
| raw + history embedding | raw plus the embedding of the transaction with its history |
| raw + single embedding | raw plus the embedding of the transaction alone (v1's combined arm) |
| history aggregates | hand-built statistics of the same history, with no model: counts, recency, amount statistics, how new the merchant is |

The last arm is the stronger, honest reference: it shows how much of the
history's value a few lines of feature engineering already capture. The head
is XGBoost when the extra is installed, as in v1, and logistic regression
otherwise.

In [ ]:
profiles = {p.entity_id: p for p in prepared.profiles}
cfg = config.evaluation.tabular_baselines
featurizers = {
    history: TabularFeaturizer.fit(
        rows.train_sequences, profiles, top_k=cfg.top_k, history=history, exclude_keys=cfg.exclude_keys
    )
    for history in (False, True)
}
raw_tr, raw_ev = (np.nan_to_num(featurizers[False].transform(s, profiles)) for s in (rows.train_sequences, rows.eval_sequences))
agg_tr, agg_ev = (np.nan_to_num(featurizers[True].transform(s, profiles)) for s in (rows.train_sequences, rows.eval_sequences))
head = "xgboost" if extra_available("xgboost") else "logistic_regression"


def arms(emb):
    return {
        "raw": (raw_tr, raw_ev),
        "raw + history embedding": (np.hstack([raw_tr, emb["history"]]), np.hstack([raw_ev, emb["history_eval"]])),
        "raw + single embedding": (np.hstack([raw_tr, emb["single"]]), np.hstack([raw_ev, emb["single_eval"]])),
        "history aggregates": (agg_tr, agg_ev),
    }


def score(emb, seed):
    return {
        name: classification_metrics(
            rows.y_eval, fit_predict_probe(x_tr, rows.y_train, x_ev, seed=seed, head=head),
            k=config.evaluation.precision_at_k,
        )
        for name, (x_tr, x_ev) in arms(emb).items()
    }


table = pd.DataFrame(score(emb, config.seed)).T[["pr_auc", "roc_auc"]]
table["AP lift over raw"] = table["pr_auc"] / table.loc["raw", "pr_auc"] - 1
print("head", head, "| test rows", len(rows.y_eval), "| test fraud", int(rows.y_eval.sum()))
table

On the generated book every arm can score perfectly: its fraud is easy to
spot and there are only a handful of test rows. The numbers there only show
that the wiring works; the real comparison is the recorded TabFormer run in
section 4.

## 2. One run is not a result

With few positives, scores move with the random seed. v1 reported one seed.
Here the seed changes pretraining and the classifier; the rows stay fixed.
Before believing a gap between two arms, look at the spread.

In [ ]:
runs = {}
for seed in range(3):
    runs[seed] = {name: m["pr_auc"] for name, m in score(embed(pretrain(seed)), seed).items()}
spread = pd.DataFrame(runs).T
spread.plot(kind="box", figsize=(7, 3), title="Test AP over 3 seeds", rot=15)
plt.show()
spread.describe().loc[["min", "50%", "max"]]

The history-aggregate arm doesn't depend on pretraining, so it moves only
with the classifier's seed. The embedding arms can move enough to reverse the
ranking of section 1.

## 3. The recipe's version

The recipe scores the same four arms on the same kind of rows, with two
differences: every XGBoost arm uses one fixed setting
(`tabular_baselines.xgboost_params`), so arms differ only in their features,
and the evaluation side scores every test-period transaction of its users. Its
`metrics.json` keys are `history_window_xgboost_raw`,
`history_window_xgboost_raw_embedding`,
`history_window_xgboost_raw_embedding:single_event`, and
`history_window_tabular_xgboost`.

```bash
uv run tfm report --recipe blueprint_v1 \
  --config recipes/blueprint_v1/config.yaml --device cuda \
  --run-dir outputs/blueprint_v1/gb10
```

To report a range, `benchmarks/seed_range.py` repeats the whole run at several
seeds; each seed moves pretraining, the cus## 4. On the real book

Three full-size seeds are recorded
([`blueprint_v1_seeds.md`](../../benchmarks/results/blueprint_v1_seeds.md)):
all TabFormer users, the 25.9M-parameter Llama, and every test transaction of
the evaluation users, about 700,000 rows with 801–845 frauds per seed.

| arm | lift over raw, seeds 42 / 43 / 44 |
|---|---:|
| raw + history embedding | −1% / +42% / +4% |
| raw + single embedding | +146% / +145% / +192% |
| history aggregates | +23% / +57% / +66% |

The pretrained embedding adds a lot to the raw features, as v1 claimed, but
it is the embedding of the transaction alone that does it. Read at the end of
128 transactions, the history embedding's lift changes from seed to seed,
although the hand-built aggregates show the history does carry signal. Part
of the single embedding's advantage is the raw row's weakness: it one-hot
encodes only the ten most common merchants and MCCs, and most test frauds
fall outside them. The record lists what would separate the two.


> **What changed from v1**
>
> - **The transaction with its history.** v1's combined arm used an embedding of
>   the transaction alone. Here the headline arm embeds it with the customer's
>   recent history, and v1's embedding is kept as a control.
> - **A seed range, not one number.** v1's published lift came from one seed.
> - **Every arm tuned the same way, no identifiers.** v1 hand-tuned each arm and
>   fed the user id to its raw arm.
> - **A model that predicts rather than copies.** The embeddings come from a
>   correctly shifted objective (notebook 03).

## Takeaways

- Compare feature sets with the head held fixed.
- Report the spread over seeds; with ~100 test frauds one seed can reverse a
  comparison.
- Embeddings earn their place by improving the combined model, and the
  single-transaction control shows how much of that comes from the history.

This ends the Blueprint v1 series. [Blueprint v2](../pragmatiq)
rebuilds the model slot by slot with profiles and key-value-time tokens.
Recipe: `uv run tfm evaluate --recipe blueprint_v1`.